## 1. Carga de la instancia

Cargamos el archivo `instancia_laguna.json` generado en el notebook `01_preparacion_datos_laguna.ipynb`. Este JSON es el **único input** que necesita el modelo de optimización.

Se extraen y tipan explícitamente:
- **`conjunto_I`**: diccionario de edificios (puntos de demanda)
- **`conjunto_J`**: diccionario de ubicaciones candidatas para contenedores
- **`conjunto_K`**: tipos de residuo `{0: orgánica, 1: resto, 2: reciclable, 3: peligrosos}`
- **`distancia`**: matriz dispersa $d_{ij}$ — solo contiene los pares $(i, j)$ con $d_{ij} \leq 350$ m
- **Parámetros del modelo**: $r_k$, $c_k$, $Q_k$, $C_j$, $N_j$, $r_0$

> **Nota sobre los IDs**: Los edificios de `conjunto_I` conservan su ID original de OpenStreetMap en formato string (ej. `"('way', 78313523)"`). Los candidatos de `conjunto_J` son strings numéricos (ej. `"262190674"`). Ambos se mantienen como `str` para consistencia.

In [15]:
import json
from collections import defaultdict
from typing import Dict, List, Tuple, Any

RUTA_JSON: str = "data/processed/instancia_laguna.json"

instancia: Dict[str, Any] = {}

with open(RUTA_JSON, "r") as file:
  instancia = json.load(file)

# --- Extracción de datos ---
params: Dict[str, Any] = instancia["parametros"]

conjunto_K: Dict[int, str] = {int(k): v for k, v in instancia["conjunto_K"].items()}
conjunto_I: Dict[str, Dict] = {ed["id"]: ed for ed in instancia["conjunto_I"]}
conjunto_J: Dict[str, Dict] = {cn["id"]: cn for cn in instancia["conjunto_J"]}

distancia: Dict[str, Dict[str, float]] = {
    i: {j: d for j, d in vecinos.items()}
    for i, vecinos in instancia["distancias_dij"].items()
}

r_k: Dict[int, float] = {int(k): v for k, v in params["r_k"].items()}
c_k: Dict[int, float] = {int(k): v for k, v in params["c_k"].items()}
Q_k: Dict[int, float] = {int(k): v for k, v in params["Q_k"].items()}
C_j: int = params["C_j"]
N_j: int = params["N_j"]
r_0: float = params["r_0"]

print(f"Instancia cargada con éxito:")
print(f"   |I| = {len(conjunto_I)} edificios")
print(f"   |J| = {len(conjunto_J)} candidatos")
print(f"   |K| = {len(conjunto_K)} tipos de residuos")
print(f"   Conexiones totales en d_ij: {sum(len(v) for v in distancia.values())}")
print(f"   Radios r_k: {r_k}")
print(f"   Costos c_k: {c_k}")
print(f"   Capacidades Q_k: {Q_k}")
print(f"   Costo fijo C_j: {C_j}")
print(f"   Número máximo de candidatos N_j: {N_j}")
print(f"   Radio inicial r_0: {r_0}")

Instancia cargada con éxito:
   |I| = 337 edificios
   |J| = 117 candidatos
   |K| = 4 tipos de residuos
   Conexiones totales en d_ij: 12839
   Radios r_k: {0: 150, 1: 150, 2: 250, 3: 350}
   Costos c_k: {0: 350, 1: 300, 2: 250, 3: 500}
   Capacidades Q_k: {0: 120, 1: 120, 2: 120, 3: 120}
   Costo fijo C_j: 4000.0
   Número máximo de candidatos N_j: 8
   Radio inicial r_0: 10.0


## 2. Check de factibilidad por tipo de residuo

Antes de construir el modelo en Gurobi, verificamos que la instancia es **matemáticamente factible**.

La restricción (2) del modelo exige que cada edificio $i$ sea asignado a exactamente un punto $j$ para cada tipo de residuo $k$. Esto solo es posible si existe al menos un candidato $j$ a distancia $d_{ij} \leq r_k$ para cada par $(i, k)$.

> ⚠️ El check de huérfanos del notebook 01 solo garantizaba $d_{ij} \leq 350$ m (cutoff global de Dijkstra). Este check es más estricto: verifica cada radio $r_k$ por separado.

| Tipo | Radio $r_k$ |
|------|------------|
| Orgánica | 150 m |
| Resto | 150 m |
| Reciclable | 250 m |
| Peligrosos | 350 m |

Si algún edificio queda sin cobertura para algún tipo, Gurobi devolverá `INFEASIBLE` sin información útil.

In [ ]:
huerfanos: Dict[int, List[str]] = defaultdict(list)

for i, vecinos in distancia.items():
    for k, radio in r_k.items():
        j_validos: List[str] = [j_id for j_id, dist in vecinos.items() if dist <= radio]
        if not j_validos:
            huerfanos[k].append(i)

if not huerfanos:
    print("\nTodos los edificios tienen al menos un candidato válido para cada tipo de residuo.")
else:
    print("\nEdificios sin candidatos válidos:")
    for k, edificios in huerfanos.items():
        print(f"  Tipo {k} (r_k={r_k[k]}): {len(edificios)} edificios sin candidatos válidos")


Todos los edificios tienen al menos un candidato válido para cada tipo de residuo.


In [1]:
import gurobipy as gp
m = gp.Model()
print("Licencia OK")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2827120
Academic license 2827120 - for non-commercial use only - registered to al___@ull.edu.es
Licencia OK
